# Experiment 4 - Native Gold Standard (Bengali)

Fine-tune and evaluate directly on **native Bengali** splits.

| Condition | Backbone | Train | Test |
|---|---|---|---|
| **4a** controlled | `FacebookAI/xlm-roberta-base` | bn-train | bn-test |
| **4b** specialized | `csebuetnlp/banglabert` | bn-train | bn-test |

Both conditions use the **same stratified splits** (seed 42), so the only
difference is the backbone. 4a is the controlled condition comparable to
configs 1-3; 4b is the native-specialized oracle and is reported as
non-controlled.

Each condition is run with 3 seeds (42, 43, 44).
Metrics: macro-F1 (primary), per-class F1, QWK, MAE, and the
upgrade/ok/downgrade error-direction matrix.

Pinned `transformers==4.57.6`, which is also the version Colab ships on the
2026.01 runtime.

In [ ]:
# Runtime check: this experiment needs a GPU.
import torch, transformers

print("torch        :", torch.__version__)
print("transformers :", transformers.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device       :", torch.cuda.get_device_name(0))
else:
    raise RuntimeError(
        "No GPU found. In Colab go to Runtime > Change runtime type > "
        "Hardware accelerator = GPU, then re-run this cell."
    )

In [ ]:
# Install pinned deps. torch is deliberately NOT installed so Colab's
# CUDA build stays intact.
!pip install -q "transformers==4.57.6" accelerate openpyxl pandas scikit-learn

In [ ]:
# Get the project. Idempotent: safe to re-run after a runtime restart.
# The datasets are tracked in the repo, so the clone brings code + data.
import os, subprocess, pathlib

if not pathlib.Path("NLP").exists():
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/iammatthi/NLP.git"], check=True)
os.chdir("NLP")
print("working dir:", os.getcwd())

data_file = pathlib.Path("data/raw/depression-severity-bengali.xlsx")
assert data_file.exists(), f"{data_file} missing - the clone did not bring the data"
print("dataset OK:", data_file, f"({data_file.stat().st_size:,} bytes)")

In [ ]:
# CPU-only build check: verifies both backbones load and produce (n, 4) logits.
# Exits non-zero on failure.
!python scripts/smoke_test_model.py

In [ ]:
# Stratified 80/10/10 splits, fixed seed. Regenerated every session; the seed
# makes them byte-reproducible so runs stay comparable.
!python scripts/prepare_splits.py --lang bn --seed 42

## Train

Class-weighted cross-entropy is on by default (inverse-frequency from the train
split) to counteract the label prior without resampling. Checkpoints are
selected on validation macro-F1; the best one is written to `<output-dir>/final/`.

In [ ]:
# Condition 4a - XLM-R on native Bengali (controlled)
!python train.py --model xlmr --lang bn --output-dir outputs/4a_xlmr_seed42 --seed 42 --epochs 4 --train-batch-size 16 --learning-rate 2e-5 --bf16
!python train.py --model xlmr --lang bn --output-dir outputs/4a_xlmr_seed43 --seed 43 --epochs 4 --train-batch-size 16 --learning-rate 2e-5 --bf16
!python train.py --model xlmr --lang bn --output-dir outputs/4a_xlmr_seed44 --seed 44 --epochs 4 --train-batch-size 16 --learning-rate 2e-5 --bf16

In [ ]:
# Condition 4b - BanglaBERT on native Bengali (specialized)
!python train.py --model banglabert --lang bn --output-dir outputs/4b_banglabert_seed42 --seed 42 --epochs 4 --train-batch-size 16 --learning-rate 2e-5 --bf16
!python train.py --model banglabert --lang bn --output-dir outputs/4b_banglabert_seed43 --seed 43 --epochs 4 --train-batch-size 16 --learning-rate 2e-5 --bf16
!python train.py --model banglabert --lang bn --output-dir outputs/4b_banglabert_seed44 --seed 44 --epochs 4 --train-batch-size 16 --learning-rate 2e-5 --bf16

## Evaluate on the held-out test split

In [ ]:
!python evaluate.py --model-dir outputs/4a_xlmr_seed42/final --lang bn --split test --seed 42
!python evaluate.py --model-dir outputs/4a_xlmr_seed43/final --lang bn --split test --seed 42
!python evaluate.py --model-dir outputs/4a_xlmr_seed44/final --lang bn --split test --seed 42
!python evaluate.py --model-dir outputs/4b_banglabert_seed42/final --lang bn --split test --seed 42
!python evaluate.py --model-dir outputs/4b_banglabert_seed43/final --lang bn --split test --seed 42
!python evaluate.py --model-dir outputs/4b_banglabert_seed44/final --lang bn --split test --seed 42

## Summarise across seeds

In [ ]:
import glob, json, statistics as st

def mean_std(vals):
    m = st.mean(vals)
    s = st.stdev(vals) if len(vals) > 1 else 0.0
    return f"{m:.4f} +/- {s:.4f}"

rows = []
for tag in ("4a_xlmr", "4b_banglabert"):
    runs = [json.load(open(p)) for p in sorted(glob.glob(f"outputs/{tag}_seed*/final/eval_test.json"))]
    if not runs:
        print(f"{tag}: no results"); continue
    row = {"condition": tag, "n_seeds": len(runs)}
    for key in ("macro_f1", "qwk", "mae"):
        row[key] = mean_std([r[key] for r in runs])
    row["downgrade_rate"] = mean_std([r["direction"]["downgrade_rate"] for r in runs])
    row["upgrade_rate"] = mean_std([r["direction"]["upgrade_rate"] for r in runs])
    rows.append(row)

for row in rows:
    print(f"{row['condition']:16s} n={row['n_seeds']}")
    for k in ("macro_f1", "qwk", "mae", "downgrade_rate", "upgrade_rate"):
        print(f"  {k:16s} {row[k]}")

# per-class F1 for the paper table
print("\nper-class F1 (mean across seeds):")
for tag in ("4a_xlmr", "4b_banglabert"):
    runs = [json.load(open(p)) for p in sorted(glob.glob(f"outputs/{tag}_seed*/final/eval_test.json"))]
    if not runs: continue
    print(f"  {tag}:")
    for cls in ("minimal", "mild", "moderate", "severe"):
        print(f"    {cls:9s} {mean_std([r[cls] for r in runs])}")

In [ ]:
# Export results. Checkpoints are large and are NOT included.
# Paths are preserved inside the zip so each run's results stay identifiable.
import glob, zipfile

paths = sorted(
    glob.glob("outputs/*/final/eval_test.json")
    + glob.glob("outputs/*/train_metrics.json")
)
assert paths, "no results found - run the training and evaluation cells first"

with zipfile.ZipFile("results.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for p in paths:
        z.write(p)

print(f"zipped {len(paths)} files:")
for p in paths:
    print(" ", p)

from google.colab import files
files.download("results.zip")
print("downloaded results.zip")

## Notes

- **Expected runtime:** ~25-45 min per training run on a T4 (bn-train is 3917
  rows, max_len 256). Six runs = roughly 2.5-4.5 hours total. Checkpoints are
  saved to `outputs/` on the VM, which is wiped when the runtime recycles -
  so run the export cell (or re-run from scratch) before the session dies.
- **Re-running after a runtime restart:** everything is scripted, so just
  re-run all cells. Cell 4 re-clones; cell 6 regenerates identical splits.
- **Adding a seed:** extend the `--seed` list in the training cells
  (e.g. add 45) and the corresponding evaluate command.
- **Ablation:** `--no-class-weights` disables the weighted loss.
- Full replication commands, dataset provenance and metrics rationale are in
  `README.md` -> *Replicating experiment 4*.